In [ ]:
FILES = {"config.py": "\"\"\"Global configuration: paths and puzzle constants.\"\"\"\nimport os\n\n# ---- Puzzle geometry ----\nGRID = 24            # 24x24 fragments\nFS = 20              # fragment size (px)\nIMG = GRID * FS      # 480\nNFRAG = GRID * GRID  # 576\n\n# ---- Paths (big data on E:, code on C:) ----\nDATA_ROOT = os.environ.get(\"PAZZLE_DATA\", r\"E:/pazzle_data\")\nWORK_ROOT = os.environ.get(\"PAZZLE_WORK\", r\"E:/pazzle_work\")\n\nTRAIN_INP = os.path.join(DATA_ROOT, \"train\", \"inputs\")\nTRAIN_TGT = os.path.join(DATA_ROOT, \"train\", \"targets\")\nTEST_DIR  = os.path.join(DATA_ROOT, \"test\")\n\nCKPT_DIR  = os.path.join(WORK_ROOT, \"ckpt\")\nCACHE_DIR = os.path.join(WORK_ROOT, \"cache\")\nSUB_DIR   = os.path.join(WORK_ROOT, \"submissions\")\nfor d in (CKPT_DIR, CACHE_DIR, SUB_DIR):\n    os.makedirs(d, exist_ok=True)\n\n# ---- Distortion params (from Task.txt) ----\nBRIGHT = 30.0        # +/- additive\nCONTRAST = (0.70, 1.30)\nNOISE_SIGMA = (40.0, 55.0)\nJPEG_Q = (35, 50)\nBLUR_KSIZE = 3       # Gaussian 3x3\n\n# ---- Held-out split for local validation ----\nVAL_COUNT = 300      # last N train names reserved for local eval\nSEED = 1234\n", "restore_tile.py": "\"\"\"Pre-assembly tile restoration: dirty 20x20 fragment -> clean 20x20 fragment.\n\nWhy this exists\n---------------\nMeasured seam-matching budget (best-buddy precision, what a solver needs):\n\n    clean tiles                        0.946\n    clean + intra-tile 3x3 blur        0.863\n    dirty + ORACLE photometry          0.295\n    dirty as-is                        0.113\n\nSo the ceiling is not the matcher, it is the input.  Every scorer in this repo\nwas fitted on the 0.113 row.  This module attacks the row itself.\n\nDesign\n------\nThe generator applies, per fragment, a SCALAR affine  x -> a*(x-pivot)+pivot+b\nbefore noise/blur/JPEG.  Normalising a tile by its own mean/std therefore\nremoves a and b exactly.  The trunk runs in that normalised space and only has\nto recover STRUCTURE; a separate global head re-predicts the absolute mean/std.\nMixing the two in one conv stack makes the network fight itself, because b is\nnot recoverable from tile content while structure is.\n\nThe restoration target is the ORIGINAL tile, not a denoised one: undoing the\n3x3 blur is worth 0.863 -> 0.946 of solver headroom.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom config import GRID as G, FS, NFRAG as N\n\n\n# --------------------------------------------------------------------------- #\n# tiles\n# --------------------------------------------------------------------------- #\ndef to_frags(img: np.ndarray) -> np.ndarray:\n    \"\"\"(480,480,3) -> (576,20,20,3), row-major grid order.\"\"\"\n    return img.reshape(G, FS, G, FS, 3).transpose(0, 2, 1, 3, 4).reshape(N, FS, FS, 3)\n\n\ndef from_frags(frags: np.ndarray) -> np.ndarray:\n    \"\"\"(576,20,20,3) -> (480,480,3).\"\"\"\n    return frags.reshape(G, G, FS, FS, 3).transpose(0, 2, 1, 3, 4).reshape(G * FS, G * FS, 3)\n\n\ndef blur3_np(x: np.ndarray) -> np.ndarray:\n    \"\"\"Separable 3x3 Gaussian with reflect padding, matching the generator.\"\"\"\n    xp = np.pad(x, ((0, 0), (1, 1), (0, 0), (0, 0)), \"reflect\")\n    x = .25 * xp[:, :-2] + .5 * xp[:, 1:-1] + .25 * xp[:, 2:]\n    xp = np.pad(x, ((0, 0), (0, 0), (1, 1), (0, 0)), \"reflect\")\n    return .25 * xp[:, :, :-2] + .5 * xp[:, :, 1:-1] + .25 * xp[:, :, 2:]\n\n\n# --------------------------------------------------------------------------- #\n# model\n# --------------------------------------------------------------------------- #\nclass ResBlock(nn.Module):\n    def __init__(self, ch: int, dilation: int = 1):\n        super().__init__()\n        self.c1 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation, padding_mode=\"reflect\")\n        self.c2 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation, padding_mode=\"reflect\")\n        self.n1 = nn.GroupNorm(8, ch)\n        self.n2 = nn.GroupNorm(8, ch)\n\n    def forward(self, x):\n        h = F.gelu(self.n1(self.c1(x)))\n        return x + self.n2(self.c2(h))\n\n\nclass TileRestorer(nn.Module):\n    \"\"\"dirty tile -> clean tile.  Full resolution throughout (20x20 is too small\n    to pool), receptive field widened with dilation instead of downsampling.\"\"\"\n\n    def __init__(self, ch: int = 96, blocks: int = 6, residual: bool = False,\n                 checkpoint: bool = False, ycc: bool = False):\n        super().__init__()\n        # Predict a CORRECTION to the normalised input rather than the structure\n        # from scratch, so the network starts from identity.  Adds no weights,\n        # hence checkpoints stay loadable; older ones were trained without it and\n        # must keep residual=False.\n        self.residual = residual\n        # Recompute block activations in the backward pass.  A 576-tile board\n        # at ch=128/blocks=8 otherwise exceeds the 8 GB card and silently spills\n        # into WDDM shared memory, where training emits no step at all.\n        self.checkpoint = checkpoint\n        # Extra input planes carrying the YCrCb view.  JPEG quantises chroma\n        # separately and subsamples it 4:2:0, so the colour channels survive the\n        # corruption far better: measured residual sigma is 11.68 on Y against\n        # 5.52 and 4.85 on Cr and Cb.  In RGB that advantage is smeared across\n        # all three planes, so the split is handed to the network explicitly.\n        self.ycc = ycc\n        # 4 input planes: normalised RGB + a constant plane carrying tile std\n        self.stem = nn.Conv2d(7 if ycc else 4, ch, 3, padding=1, padding_mode=\"reflect\")\n        dil = [1, 2, 3, 2, 1, 1][:blocks] + [1] * max(0, blocks - 6)\n        self.body = nn.Sequential(*[ResBlock(ch, d) for d in dil])\n        self.head = nn.Conv2d(ch, 3, 3, padding=1, padding_mode=\"reflect\")\n        # global head: predicts the CLEAN tile's mean/std from pooled features\n        # plus the observed statistics (b is only partly recoverable, so this\n        # head learns the Bayes shrinkage rather than an exact inverse).\n        self.glob = nn.Sequential(\n            nn.Linear(ch + 6, 128), nn.GELU(),\n            nn.Linear(128, 128), nn.GELU(),\n            nn.Linear(128, 6),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"x: (B,3,20,20) float in [0,255].  Returns (B,3,20,20) in [0,255].\"\"\"\n        mean = x.mean(dim=(2, 3), keepdim=True)\n        std = x.std(dim=(2, 3), keepdim=True).clamp_min(1e-3)\n        xn = (x - mean) / std\n        std_plane = (std / 64.0).expand(-1, -1, FS, FS)[:, :1]\n        planes = [xn, std_plane]\n        if self.ycc:\n            r, g, bl = x[:, 0:1], x[:, 1:2], x[:, 2:3]\n            y = 0.299 * r + 0.587 * g + 0.114 * bl\n            cr = (r - y) * 0.713\n            cb = (bl - y) * 0.564\n            yc = torch.cat([y, cr, cb], dim=1)\n            yc = (yc - yc.mean(dim=(2, 3), keepdim=True)) / yc.std(dim=(2, 3), keepdim=True).clamp_min(1e-3)\n            planes.append(yc)\n        h = self.stem(torch.cat(planes, dim=1))\n        if self.checkpoint and self.training:\n            from torch.utils.checkpoint import checkpoint_sequential\n            h = checkpoint_sequential(self.body, len(self.body), h, use_reentrant=False)\n        else:\n            h = self.body(h)\n        struct = self.head(h)                                    # normalised structure\n        if self.residual:\n            struct = struct + xn\n\n        pooled = h.mean(dim=(2, 3))\n        stats = torch.cat([mean.flatten(1) / 128.0, std.flatten(1) / 64.0], dim=1)\n        g = self.glob(torch.cat([pooled, stats], dim=1))\n        # predict clean stats as a correction of the observed ones\n        out_mean = mean.flatten(1) + 30.0 * torch.tanh(g[:, :3])\n        out_std = std.flatten(1) * torch.exp(g[:, 3:].clamp(-1.5, 1.5))\n\n        struct = struct - struct.mean(dim=(2, 3), keepdim=True)\n        struct = struct / struct.std(dim=(2, 3), keepdim=True).clamp_min(1e-3)\n        return struct * out_std[:, :, None, None] + out_mean[:, :, None, None]\n\n\n# --------------------------------------------------------------------------- #\n# seam scoring on restored tiles (the gate metric)\n# --------------------------------------------------------------------------- #\ndef ridge_cost(tiles: np.ndarray, w: float = 0.03, cols: int = 3, axis: str = \"h\") -> np.ndarray:\n    \"\"\"cost[i,j] = var(d) + w*mean(d)^2 over the facing border strips.\n\n    The per-tile brightness b is a nuisance parameter: fitting it freely (w=0)\n    throws away real DC signal, ignoring it (w=1) eats the +-30 jitter, so the\n    ridge optimum sits in between.\n\n    Defaults are the held-out optimum ON RESTORED tiles (bb_prec 0.4117).  Raw\n    tiles peak elsewhere (w=0.12, cols=2, bb_prec 0.3438): restoration already\n    removes part of the brightness error, which both devalues the DC term and\n    widens the usable border strip.\n    \"\"\"\n    n_tiles = len(tiles)\n    if axis == \"h\":\n        A = tiles[:, :, -cols:, :].reshape(n_tiles, -1, 3)\n        B = tiles[:, :, :cols, :].reshape(n_tiles, -1, 3)\n    else:\n        A = tiles[:, -cols:, :, :].reshape(n_tiles, -1, 3)\n        B = tiles[:, :cols, :, :].reshape(n_tiles, -1, 3)\n    n = A.shape[1]\n    m2 = ((A ** 2).sum(1)[:, None, :] + (B ** 2).sum(1)[None, :, :]\n          - 2 * np.einsum(\"ikc,jkc->ijc\", A, B)) / n\n    mu = A.mean(1)[:, None, :] - B.mean(1)[None, :, :]\n    return (m2 - mu ** 2 + w * mu ** 2).sum(-1)\n\n\ndef ridge_cost_torch(tiles: torch.Tensor, w: float = 0.03, cols: int = 3,\n                     axis: str = \"h\") -> torch.Tensor:\n    \"\"\"Differentiable twin of ridge_cost.  tiles: (N,3,20,20) -> (N,N) cost.\n\n    Being differentiable is the point: it lets the restorer be trained directly\n    on \"make true neighbours findable\" instead of on pixel L1, which converges\n    to the conditional mean and smooths away the border microstructure that\n    carries the entire adjacency signal.\n    \"\"\"\n    if axis == \"h\":\n        a, b = tiles[:, :, :, -cols:], tiles[:, :, :, :cols]\n    else:\n        a, b = tiles[:, :, -cols:, :], tiles[:, :, :cols, :]\n    n_tiles = tiles.shape[0]\n    a = a.reshape(n_tiles, 3, -1)\n    b = b.reshape(n_tiles, 3, -1)\n    n = a.shape[2]\n    m2 = ((a ** 2).sum(2)[:, None, :] + (b ** 2).sum(2)[None, :, :]\n          - 2 * torch.einsum(\"icn,jcn->ijc\", a, b)) / n\n    mu = a.mean(2)[:, None, :] - b.mean(2)[None, :, :]\n    return (m2 - mu ** 2 + w * mu ** 2).sum(-1)\n\n\ndef seam_infonce(tiles: torch.Tensor, inv_temp: torch.Tensor, good: torch.Tensor,\n                 w: float = 0.03, cols: int = 3, metric: str = \"ridge\",\n                 hard_k: int = 0) -> torch.Tensor:\n    \"\"\"Contrastive seam loss over a whole board given in TRUE grid order.\n\n    `good` is the per-position label-confidence mask.  A row counts only when\n    BOTH its anchor and its true neighbour are confidently placed: the Hungarian\n    matcher is 0.825 accurate overall, so unmasked rows would teach the model\n    adjacencies that do not exist.\n    \"\"\"\n    total = tiles.new_zeros(())\n    n_used = 0\n    metrics = (\"ridge\", \"mgc\") if metric == \"both\" else (metric,)\n    for axis, step in ((\"h\", 1), (\"v\", G)):\n      for met in metrics:\n          if met == \"mgc\":\n              # Train on the measure we will actually score with.  MGC reads\n              # gradients across the seam and reaches bb_prec 0.994 on clean tiles\n              # versus 0.796 for the ridge cost, but it is destroyed by residual\n              # noise, so the restorer must be optimised for it directly.\n              # Both are kept because they dominate in different noise regimes:\n              # on real restored tiles ridge scores 0.396 and MGC 0.113, while on\n              # clean tiles MGC wins 0.994 to 0.796.\n              from mgc import mgc_cost_torch\n              cost = mgc_cost_torch(tiles, axis)\n          else:\n              cost = ridge_cost_torch(tiles, w, cols, axis)\n          # Row-standardise before the temperature: raw ridge costs run in the\n          # thousands, so an unnormalised scale drives softmax to one-hot and the\n          # loss sits far above ln(576) no matter what the model does.\n          cost = (cost - cost.mean(1, keepdim=True)) / cost.std(1, keepdim=True).clamp_min(1e-6)\n          logits = -cost * inv_temp.exp()\n          logits = logits - torch.eye(len(tiles), device=tiles.device) * 1e4   # mask self\n          on_grid = torch.tensor(\n              [((p % G) != G - 1 if axis == \"h\" else p < N - G) for p in range(N)],\n              device=tiles.device)\n          idx = torch.nonzero(on_grid & (good > 0) & torch.roll(good > 0, -step), as_tuple=True)[0]\n          if len(idx) == 0:\n              continue\n          if hard_k and hard_k < len(tiles) - 1:\n              # Focus on the confusable tail.  Averaged over all 575 negatives\n              # the loss is dominated by tiles that are trivially far away and\n              # contribute almost no gradient; restricting it to the hardest few\n              # spends every step on the competitors that actually cost us R@1.\n              rows = logits[idx]\n              keep = rows.topk(hard_k, dim=1).indices\n              keep = torch.cat([(idx + step).unsqueeze(1), keep], dim=1)\n              sub = torch.gather(rows, 1, keep)\n              total = total + F.cross_entropy(\n                  sub, torch.zeros(len(idx), dtype=torch.long, device=tiles.device))\n          else:\n              total = total + F.cross_entropy(logits[idx], idx + step)\n          n_used += 1\n\n    return total / max(1, n_used)\n\n\ndef seam_metrics(tiles: np.ndarray, w: float = 0.03, cols: int = 3,\n                 metric: str = \"ridge\") -> dict[str, float]:\n    \"\"\"tiles must be in TRUE grid order.  Reports R@1/R@20/best-buddy precision.\n\n    With metric=\"both\" the two measures are reported side by side and the\n    headline figure is the better of them, since the solver is free to use\n    whichever wins: ridge dominates on noisy tiles (0.396 vs 0.113) and MGC on\n    clean ones (0.994 vs 0.796).\n    \"\"\"\n    if metric == \"both\":\n        a = seam_metrics(tiles, w, cols, \"ridge\")\n        b = seam_metrics(tiles, w, cols, \"mgc\")\n        out = {f\"ridge_{k}\": v for k, v in a.items()}\n        out.update({f\"mgc_{k}\": v for k, v in b.items()})\n        for k in (\"R1\", \"R20\", \"bb_prec\"):\n            out[k] = max(a[k], b[k])\n        return out\n    out = {}\n    for axis, step, edge in ((\"h\", 1, lambda p: (p % G) != G - 1),\n                             (\"v\", G, lambda p: p < N - G)):\n        if metric == \"mgc\":\n            from mgc import mgc_cost\n            C = mgc_cost(tiles, axis)\n        else:\n            C = ridge_cost(tiles, w, cols, axis)\n        np.fill_diagonal(C, np.inf)\n        rows = np.array([p for p in range(N) if edge(p)])\n        order = np.argsort(C[rows], axis=1)\n        rank = np.array([np.where(order[k] == rows[k] + step)[0][0] for k in range(len(rows))])\n        best_f, best_b = np.argmin(C, 1), np.argmin(C, 0)\n        bb = [(i, best_f[i]) for i in range(N) if best_b[best_f[i]] == i]\n        ok = sum(1 for i, j in bb if edge(i) and j == i + step)\n        out[f\"R1_{axis}\"] = float((rank == 0).mean())\n        out[f\"R20_{axis}\"] = float((rank < 20).mean())\n        out[f\"bb_{axis}\"] = float(ok / max(1, len(bb)))\n    out[\"R1\"] = 0.5 * (out[\"R1_h\"] + out[\"R1_v\"])\n    out[\"R20\"] = 0.5 * (out[\"R20_h\"] + out[\"R20_v\"])\n    out[\"bb_prec\"] = 0.5 * (out[\"bb_h\"] + out[\"bb_v\"])\n    return out\n", "choose5.py": "\"\"\"A model that sees five candidate seams at once and picks one.\n\nWhy this shape, and not another re-ranker\n-----------------------------------------\nM409 leaves one door and M407 sizes it. The shipping roster finds the true\npartner first 0.299 of the time and inside its top five 0.475 of the time; the\npercolation knee is at 450 to 500 correct bonds and 0.475 of 1104 is 524. So\nchoosing perfectly inside the top five solves the board. We currently choose\nright 66 per cent of the time there and need 86.\n\nEverything that has failed at this task shares one property: it scored\ncandidates ONE AT A TIME and compared the numbers afterwards. The matcher does,\nthe selector does, M157 and M164's re-rankers did, M404's best-square-per-\nfragment did. A margin between two scores is a SUMMARY of a comparison; the\ncomparison itself -- five seams side by side, where one continues a line and\nfour merely match colour -- has never been shown to a model.\n\nAnd the other family is closed from the opposite side. M410 measured that no\nglobal objective beats plain per-fragment top-1: the Hungarian assignment on\nseam scores reaches 332.4 correct bonds, square-closure search 335.6, mutual\nbest 315.7, plain top-1 348.4. Optimising over arrangements does not help, so\nthe remaining move is to make the per-fragment CHOICE better.\n\nThe design follows from that\n----------------------------\n* The five candidates go in together and attend to each other, so the score of\n  one is computed in the presence of its rivals. That is the whole point; a\n  per-candidate tower with a softmax on top would be the selector again.\n* A sixth NONE option, because the truth is outside the top five for 0.525 of\n  fragments and a model forced to choose would learn to guess. Abstaining is\n  worth more than guessing here: M409 measured that at our operating point\n  precision protects the four hundred fragments outside the block.\n* The seam patch is pixels from BOTH fragments across the join, `strip` columns\n  each side. M34 measured that the signal decays fast with distance from the\n  edge -- inset 0/1/2/3 gives R@1 0.159/0.084/0.059/0.040 -- so a wide patch\n  would mostly add noise.\n* The fused score and the rank come in as scalars beside the pixels, because\n  the model should improve on the matcher rather than relearn it from scratch.\n\"\"\"\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nK = 5\n\n\nclass SeamEncoder(nn.Module):\n    \"\"\"(B, 3, 20, 2*strip) -> (B, dim). One candidate's join, as pixels.\"\"\"\n\n    def __init__(self, ch=48, dim=128, strip=4):\n        super().__init__()\n        self.body = nn.Sequential(\n            nn.Conv2d(3, ch, 3, padding=1), nn.GELU(),\n            nn.Conv2d(ch, ch, 3, stride=(2, 1), padding=1),\n            nn.GroupNorm(8, ch), nn.GELU(),\n            nn.Conv2d(ch, ch * 2, 3, stride=(2, 1), padding=1),\n            nn.GroupNorm(8, ch * 2), nn.GELU(),\n            nn.Conv2d(ch * 2, ch * 2, 3, padding=1), nn.GELU(),\n        )\n        self.head = nn.Linear(ch * 2, dim)\n\n    def forward(self, x):\n        h = self.body(x)\n        return self.head(h.mean((2, 3)))\n\n\nclass CrossSeam(nn.Module):\n    \"\"\"The two sides of a join attend to EACH OTHER, row by row.\n\n    Every matcher in this project is a bi-encoder: each fragment is encoded\n    alone and the two descriptors are compared by a dot product. Attention\n    between the pair is never computed, which means the comparison is fixed to\n    \"row k of A against row k of B\" -- and a seam whose content shifts a row,\n    which the corruption and the 20-pixel quantisation make common, cannot be\n    matched that way.\n\n    Here each side becomes a sequence of row tokens and the two sequences\n    cross-attend, so the model can align row k of A with row k+1 of B when the\n    content says so. M105 to M109, M157 and M164 built re-rankers, but as\n    joint scorers over pooled features rather than as cross-attention over the\n    seam, and they moved R@1 by 0.003.\n    \"\"\"\n\n    def __init__(self, ch=48, dim=128, strip=4, layers=2, heads=4):\n        super().__init__()\n        self.strip = strip\n        self.stem = nn.Sequential(\n            nn.Conv2d(3, ch, 3, padding=1), nn.GELU(),\n            nn.Conv2d(ch, ch, 3, padding=1), nn.GroupNorm(8, ch), nn.GELU())\n        self.to_tok = nn.Linear(ch * strip, dim)\n        self.side = nn.Parameter(torch.zeros(2, 1, dim))\n        self.pos = nn.Parameter(torch.zeros(1, 40, dim))\n        layer = nn.TransformerEncoderLayer(\n            dim, heads, dim * 2, dropout=0.0, batch_first=True,\n            norm_first=True, activation=\"gelu\")\n        self.mix = nn.TransformerEncoder(layer, layers)\n        self.out = nn.Linear(dim, dim)\n\n    def forward(self, x):\n        \"\"\"(B, 3, 20, 2*strip) -> (B, dim), the two halves cross-attending.\"\"\"\n        b = x.shape[0]\n        h = self.stem(x)\n        a = h[:, :, :, :self.strip].permute(0, 2, 1, 3).reshape(b, 20, -1)\n        c = h[:, :, :, self.strip:].permute(0, 2, 1, 3).reshape(b, 20, -1)\n        t = torch.cat([self.to_tok(a) + self.side[0],\n                       self.to_tok(c) + self.side[1]], 1)\n        t = t + self.pos[:, :t.shape[1]]\n        return self.out(self.mix(t).mean(1))\n\n\nclass Choose5(nn.Module):\n    \"\"\"Five joins in, one choice out, with a NONE option.\n\n    The candidates are encoded independently and then attend to one another\n    before any of them is scored, which is the one thing a per-pair model\n    cannot do.\n    \"\"\"\n\n    def __init__(self, ch=48, dim=128, strip=4, layers=2, heads=4,\n                 encoder=\"cnn\"):\n        super().__init__()\n        self.strip = strip\n        self.enc = (CrossSeam(ch, dim, strip, layers, heads)\n                    if encoder == \"cross\" else SeamEncoder(ch, dim, strip))\n        self.scalars = nn.Sequential(\n            nn.Linear(4, dim), nn.GELU(), nn.Linear(dim, dim))\n        enc_layer = nn.TransformerEncoderLayer(\n            dim, heads, dim * 2, dropout=0.0, batch_first=True,\n            norm_first=True, activation=\"gelu\")\n        self.mix = nn.TransformerEncoder(enc_layer, layers)\n        self.none = nn.Parameter(torch.zeros(1, 1, dim))\n        self.score = nn.Linear(dim, 1)\n        # zero-initialised, so an untrained model reproduces the matcher's own\n        # ranking EXACTLY and any gain is something the model found rather than\n        # something it relearned. `coarse_field` uses the same device and says\n        # why: it makes \"no effect\" unarguable.\n        nn.init.zeros_(self.score.weight)\n        nn.init.zeros_(self.score.bias)\n        self.prior = nn.Parameter(torch.tensor(1.0))\n        self.none_bias = nn.Parameter(torch.tensor(0.0))\n\n    def forward(self, patch, scalars):\n        \"\"\"patch (B, K, 3, 20, 2*strip); scalars (B, K, 4) -> logits (B, K+1).\n\n        The matcher's own margin enters as a fixed prior and the network learns\n        a correction on top, so training starts from the matcher's top-1 and\n        moves away from it only when the pixels say so.\n        \"\"\"\n        b, k = patch.shape[:2]\n        z = self.enc(patch.reshape(b * k, *patch.shape[2:])).reshape(b, k, -1)\n        z = z + self.scalars(scalars)\n        z = torch.cat([z, self.none.expand(b, 1, z.shape[-1])], 1)\n        z = self.mix(z)\n        delta = self.score(z).squeeze(-1)\n        base = torch.cat([scalars[..., 1] * self.prior,\n                          self.none_bias.expand(b, 1)], 1)\n        return base + delta\n\n\ndef seam_patch(tiles, src, dst, axis, strip=4):\n    \"\"\"The join between two fragments, as one image.\n\n    `tiles` is (N, 20, 20, 3). For a horizontal join the last `strip` columns of\n    the source meet the first `strip` of the destination; a vertical join is the\n    same picture transposed, so one encoder serves both.\n    \"\"\"\n    a, b = tiles[src], tiles[dst]\n    if axis == \"h\":\n        p = torch.cat([a[:, :, -strip:], b[:, :, :strip]], 2)\n    else:\n        p = torch.cat([a[:, -strip:, :], b[:, :strip, :]], 1)\n        p = p.transpose(1, 2)\n    return p.permute(0, 3, 1, 2).contiguous()\n\n\ndef choose_loss(logits, label, none_weight=0.3):\n    \"\"\"Cross-entropy over the five candidates plus NONE, with NONE discounted.\n\n    `label` is the index of the true candidate, or K when the truth is outside\n    the shortlist; rows with no true partner at all are dropped by the caller.\n\n    The discount is not a detail. NONE is the correct answer for 47 per cent of\n    fragments, so plain cross-entropy is minimised by abstaining often, and an\n    abstention scores zero correct bonds -- the quantity M395 and M407 say\n    converts. The first run of this measured it: a model that starts at the\n    matcher's own 347.3 correct bonds falls to 275.9 after one epoch of the\n    undiscounted loss. At weight 0 the model is trained only where the truth is\n    in the shortlist, which is the pure five-way question.\n    \"\"\"\n    w = torch.ones(logits.shape[1], device=logits.device)\n    w[K] = none_weight\n    return F.cross_entropy(logits, label, weight=w)\n", "train_choose5.py": "\"\"\"Train the five-candidate chooser, and score it in correct bonds.\n\nThe target, from M407 and M409: 450 to 500 correct bonds a board crosses the\npercolation knee, the shipping roster's plain top-1 delivers 330 to 348, and the\ntop-five shortlist holds 524. So the number to watch is not accuracy but CORRECT\nBONDS after the choice, and the baseline to beat is the matcher's own top-1 on\nthe same boards.\n\nM306 fixed the standard of proof: two runs of the same configuration differ by\n0.028 in R@1, so nothing below about 0.03 counts, and a single seed proves\nnothing. That is why the baseline here is recomputed on the same held-out boards\nrather than quoted, and why the run prints the per-board spread.\n\"\"\"\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nimport cv2\n\nfrom choose5 import K, Choose5, choose_loss, seam_patch\nfrom config import CKPT_DIR, GRID as G, TRAIN_INP\nfrom restore_tile import to_frags\n\nN = G * G\n\n\ndef load_rgb(path):\n    \"\"\"Read one board. Defined here rather than imported from\n    `infer_coarse_field`, whose module chain pulls in the coarse-field model\n    for the sake of four lines and broke the Kaggle run.\"\"\"\n    img = cv2.imread(str(path), cv2.IMREAD_COLOR)\n    if img is None:\n        raise RuntimeError(f\"unreadable: {path}\")\n    return np.ascontiguousarray(img[:, :, ::-1])\n\n\nclass _NoCache(dict):\n    \"\"\"A dict that forgets, so the cached and uncached paths share one code.\"\"\"\n\n    def __setitem__(self, k, v):\n        pass\n\n    def get(self, k, default=None):\n        return default\n\n\nclass Boards(Dataset):\n    \"\"\"One board per item: its tiles and both directions' shortlists.\"\"\"\n\n    def __init__(self, files, cache=False):\n        self.files = files\n        self._cache = {} if cache else _NoCache()\n\n    def __len__(self):\n        return len(self.files)\n\n    def __getitem__(self, k):\n        hit = self._cache.get(k)\n        if hit is None:\n            z = np.load(self.files[k])\n            # cached as UINT8 and converted on the way out: float32 tiles are\n            # 2.76 MB a board, which is 8.7 GB over the 3141 dumps M412 says\n            # this experiment needs, and uint8 is 2.2 GB\n            tiles = to_frags(load_rgb(Path(TRAIN_INP) / str(z[\"name\"])))[\n                z[\"inv\"].astype(np.int64)].astype(np.uint8)\n            hit = (tiles,\n                   {t: (torch.from_numpy(z[f\"{t}_idx\"].astype(np.int64)),\n                        torch.from_numpy(z[f\"{t}_val\"]),\n                        torch.from_numpy(z[f\"{t}_lab\"].astype(np.int64)))\n                    for t in (\"h\", \"v\")})\n            self._cache[k] = hit\n        tiles, packs = hit\n        return torch.from_numpy(tiles.astype(np.float32)), packs\n\n\ndef collate(batch):\n    return batch\n\n\ndef board_batch(tiles, idx, val, lab, strip, dev):\n    \"\"\"Every fragment that HAS a true partner, as one batch of shortlists.\"\"\"\n    keep = (lab >= 0).nonzero(as_tuple=True)[0]\n    src = keep.repeat_interleave(K)\n    dst = idx[keep].reshape(-1)\n    return keep, src, dst, val[keep], lab[keep]\n\n\ndef run_board(model, tiles, packs, strip, dev, train, none_weight=0.3):\n    loss_sum, rows = 0.0, []\n    for axis in (\"h\", \"v\"):\n        idx, val, lab = packs[axis]\n        idx, val, lab = idx.to(dev), val.to(dev), lab.to(dev)\n        keep, src, dst, v, y = board_batch(tiles, idx, val, lab, strip, dev)\n        if not len(keep):\n            continue\n        patch = seam_patch(tiles, src, dst, axis, strip).reshape(\n            len(keep), K, 3, 20, 2 * strip)\n        rank = torch.arange(K, device=dev, dtype=torch.float32)\n        z = v - v[:, :1]\n        sc = torch.stack([v / 10.0, z, rank.expand(len(keep), K),\n                          (z == 0).float()], -1)\n        logits = model(patch, sc)\n        loss = choose_loss(logits, y, none_weight)\n        loss_sum += float(loss.detach())\n        if train:\n            loss.backward()\n        pick = logits.argmax(1)\n        rows.append((int((pick == y).sum()),\n                     int(((pick < K) & (pick == y)).sum()),\n                     int((y == 0).sum()), len(keep)))\n    return loss_sum, rows\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\"--dumps\", required=True)\n    ap.add_argument(\"--held\", type=int, default=24)\n    ap.add_argument(\"--epochs\", type=int, default=6)\n    ap.add_argument(\"--lr\", type=float, default=3e-4)\n    ap.add_argument(\"--strip\", type=int, default=4)\n    ap.add_argument(\"--ch\", type=int, default=48)\n    ap.add_argument(\"--dim\", type=int, default=128)\n    ap.add_argument(\"--layers\", type=int, default=2)\n    ap.add_argument(\"--none-weight\", type=float, default=0.3,\n                    help=\"how much the NONE class counts in the loss. It is the right answer for 47%% of fragments, so at weight 1 the model learns to abstain and an abstention is worth zero correct bonds; at 0 it is trained only where the truth is in the shortlist\")\n    ap.add_argument(\"--encoder\", default=\"cnn\",\n                    choices=(\"cnn\", \"cross\"),\n                    help=\"cnn convolves the join; cross makes the two sides of it attend to each other row by row, which no matcher in this project does -- they are all bi-encoders comparing pooled descriptors\")\n    ap.add_argument(\"--eval-every\", type=int, default=1)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--out\", default=\"choose5.pt\")\n    a = ap.parse_args()\n    torch.manual_seed(a.seed)\n    np.random.seed(a.seed)\n\n    files = sorted(Path(a.dumps).glob(\"*.npz\"))\n    if len(files) <= a.held:\n        sys.exit(f\"only {len(files)} dumps in {a.dumps}\")\n    train, held = files[a.held:], files[:a.held]\n    print(f\"{len(train)} train boards, {len(held)} held out, seed {a.seed}\",\n          flush=True)\n\n    dev = \"cuda\"\n    model = Choose5(a.ch, a.dim, a.strip, a.layers,\n                    encoder=a.encoder).to(dev)\n    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01)\n    dl = DataLoader(Boards(train, cache=True), batch_size=1,\n                    shuffle=True, collate_fn=collate, num_workers=0)\n    sched = torch.optim.lr_scheduler.OneCycleLR(\n        opt, a.lr, total_steps=max(a.epochs * len(train), 1), pct_start=0.15)\n\n    def evaluate():\n        model.eval()\n        got = base = tot = 0\n        per = []\n        with torch.no_grad():\n            for hb in held_ds:\n                tiles, packs = hb\n                tiles = tiles.to(dev)\n                _l, rows = run_board(model, tiles, packs, a.strip, dev, False)\n                g = sum(r[1] for r in rows)\n                b0 = sum(r[2] for r in rows)\n                n = sum(r[3] for r in rows)\n                got += g\n                base += b0\n                tot += n\n                per.append(g - b0)\n        model.train()\n        return got / len(held), base / len(held), tot / len(held), np.std(per)\n\n    held_ds = [Boards([f])[0] for f in held]\n    g, b0, tot, sd = evaluate()\n    print(f\"[init] chooser {g:.1f} correct bonds, matcher top-1 {b0:.1f}, \"\n          f\"{tot:.0f} fragments with a true partner\", flush=True)\n\n    for ep in range(a.epochs):\n        run = []\n        for batch in dl:\n            tiles, packs = batch[0]\n            tiles = tiles.to(dev)\n            opt.zero_grad(set_to_none=True)\n            loss, _rows = run_board(model, tiles, packs, a.strip, dev, True,\n                                    a.none_weight)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n            sched.step()\n            run.append(loss)\n        if ep % a.eval_every and ep != a.epochs - 1:\n            print(f\"[epoch {ep}] loss {np.mean(run):.4f}\", flush=True)\n            continue\n        g, b0, tot, sd = evaluate()\n        print(f\"[epoch {ep}] loss {np.mean(run):.4f}  chooser {g:.1f} against \"\n              f\"matcher {b0:.1f} correct bonds  (per-board sd {sd:.1f})\",\n              flush=True)\n\n    out = Path(CKPT_DIR) / a.out\n    torch.save({\"model\": model.state_dict(),\n                \"args\": {k: getattr(a, k) for k in\n                         (\"ch\", \"dim\", \"strip\", \"layers\", \"encoder\")}}, out)\n    print(f\"wrote {out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"}

# Everything the chooser needs, unpacked where the trainer expects it.
# Kaggle mounts a dataset at /kaggle/input/datasets/<owner>/<slug> rather than
# under the slug directly, so both searches walk instead of listing one level.
import os, sys, json, shutil
from pathlib import Path

BASE = Path("/kaggle/input")
print("mounted:", [q.name for q in BASE.iterdir()])

W = Path("/kaggle/working")
WORK = W / "pazzle_work"
(WORK / "cache").mkdir(parents=True, exist_ok=True)
(WORK / "ckpt").mkdir(parents=True, exist_ok=True)
SRC = W / "src"
SRC.mkdir(exist_ok=True)
for name, text in FILES.items():
    (SRC / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))


def find_dir(pred):
    for q in BASE.rglob("*"):
        if q.is_dir() and pred(q):
            return q
    return None


DATA = find_dir(lambda q: (q / "train" / "inputs").is_dir())
assert DATA is not None, "puzzle images not found"
DUMPS = find_dir(lambda q: q.name == "top5_new" and any(q.glob("*.npz")))
assert DUMPS is not None, "top5 dumps not found"
LABELS = next(BASE.rglob("restore_labels.npz"), None)
assert LABELS is not None, "restore_labels.npz not found"
shutil.copy(LABELS, WORK / "cache" / "restore_labels.npz")

# config.py reads these at IMPORT time and the names are PAZZLE_DATA and
# PAZZLE_WORK, not the _ROOT spellings the variables inside it use
os.environ["PAZZLE_DATA"] = str(DATA)
os.environ["PAZZLE_WORK"] = str(WORK)
print("data:", DATA)
print("dumps:", DUMPS, len(list(DUMPS.glob("*.npz"))), "boards")


In [ ]:
# M412 at twenty-three times its data. Everything else is held at the values
# M412 measured, so the only variable is the scale it named.
import numpy as np, torch, time
from pathlib import Path

sys.argv = ["train_choose5.py",
            "--dumps", str(DUMPS),
            "--held", "240",
            "--epochs", "12",
            "--ch", "64", "--dim", "192", "--layers", "3",
            "--strip", "4",
            "--none-weight", "0.3",
            "--lr", "0.0003",
            "--eval-every", "1",
            "--seed", "0",
            "--out", "/kaggle/working/choose5_big.pt"]
import train_choose5
t0 = time.time()
train_choose5.main()
print(f"done in {(time.time() - t0) / 3600:.2f} h")
